# From a specification to a governed MCP server

**Subhadip Mitra** &middot; [subhadipmitra.com](https://subhadipmitra.com) &middot; contact@subhadipmitra.com

Part of [api-mcp-compiler](https://github.com/bassrehab/api-mcp-compiler), licensed Apache-2.0.

This walks one small OpenAPI document through every stage of the compiler (ingestion,
planning, policy, the emission gate, review, code generation) and prints the actual
intermediate value at each step rather than describing it.

The specification is synthetic and deliberately awkward. Three operations carry the things
that make conversion hard: a destructive delete, three alternative security requirements, a
deprecated endpoint, and a `default` response that may or may not be a fault.

**The outputs below are verified.** `scripts/check_notebook.py` runs in the repository's
verification gate: it re-executes every code cell and fails if any output differs from what
is stored here. Nothing in this notebook is a screenshot of a version that used to work.

## 1. The source

The operation this notebook keeps returning to is the destructive one.

In [ ]:
from pathlib import Path

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
SPEC = REPO / "examples" / "openapi" / "inventory_service.yaml"

lines = SPEC.read_text(encoding="utf-8").splitlines()
found = next(i for i, line in enumerate(lines) if "purgeWarehouseItems" in line)
start = next(i for i in range(found, 0, -1) if lines[i].strip() == "delete:")

print(f"{SPEC.relative_to(REPO)}\n")
print("\n".join(lines[start : start + 12]))

examples/openapi/inventory_service.yaml

    delete:
      operationId: purgeWarehouseItems
      summary: Permanently remove every item record for a warehouse
      security:
        - inventoryOAuth: [inventory.write, inventory.admin]
        - inventoryOAuth: [inventory.write]
        - inventoryAdminKey: []
      responses:
        '204':
          description: Purged
  /warehouses/{warehouse_id}/items-v1:
    get:


## 2. Ingestion: the API Semantic IR

Ingestion normalizes OpenAPI and WSDL into one provider-independent intermediate
representation, so that everything downstream (planning, policy, generation, evaluation)
is written once against the IR rather than twice against two input formats.

The digest is of the source bytes. Every artifact produced from here carries it, which is
what makes a surface reproducible from a versioned specification and what stops a review
decision from silently carrying over to a specification that has since changed.

In [ ]:
from api_mcp_compiler.ingest.openapi import parse_openapi

ir = parse_openapi(SPEC)

print(f"{ir.service.title} {ir.service.version}, {len(ir.operations)} operations")
print(f"{ir.service.source_digest}\n")
for op in ir.operations:
    print(f"  {op.operation_id:26} {op.side_effect.value:12} idempotency={op.idempotency.value}")

Synthetic Inventory Service 2.1.0, 3 operations
sha256:3a0ae2f79c0d5b4b13e73266f61df6e77c6729e0a5d94a17716776ee9e2730a0

  listWarehouseItems         read         idempotency=idempotent
  purgeWarehouseItems        destructive  idempotency=idempotent
  listWarehouseItemsLegacy   read         idempotency=idempotent


## 3. Every field says where it came from

`derivation` separates what the document said from what the compiler worked out:

| derivation | meaning |
|---|---|
| `source` | copied from the document |
| `normalized` | the document's value, restated in the IR's vocabulary |
| `inferred` | not stated; derived from a rule, which is named |
| `default` | not stated and not inferable; a documented fallback |

`side_effect` below is `inferred`, and the rule says on what basis. That matters, because
whether an operation is destructive decides whether a human has to approve it, and a claim
that important should not be anonymous.

In [ ]:
purge = next(op for op in ir.operations if op.operation_id == "purgeWarehouseItems")

for entry in purge.provenance:
    print(f"{entry.field:12} {entry.derivation.value:10} {entry.rule}")
    print(f"{'':12} <- {entry.source_pointer}")

operation_id source     openapi.operationId
             <- openapi:#/paths/~1warehouses~1{warehouse_id}~1items/delete/operationId
protocol     normalized openapi.protocol.http
             <- openapi:#/openapi
source_pointer normalized openapi.operation.pointer
             <- openapi:#/paths/~1warehouses~1{warehouse_id}~1items/delete
route        source     openapi.paths.key
             <- openapi:#/paths/~1warehouses~1{warehouse_id}~1items
intent       source     openapi.summary
             <- openapi:#/paths/~1warehouses~1{warehouse_id}~1items/delete/summary
side_effect  inferred   openapi.side_effect.method.delete
             <- openapi:#/paths/~1warehouses~1{warehouse_id}~1items
idempotency  inferred   openapi.idempotency.rfc9110.delete
             <- openapi:#/paths/~1warehouses~1{warehouse_id}~1items
deprecated   default    openapi.operation.deprecated
             <- openapi:#/paths/~1warehouses~1{warehouse_id}~1items/delete


## 4. What it reports rather than guesses

An `Ambiguity` is recorded where the document is genuinely unclear, beside the construct
that produced it. Neither of these blocks emission; one that did, meaning `blocking=True`,
would stop the tool from being emitted executable at all.

The second one is the interesting one. Ingestion **refuses to choose** between three
alternative security requirements: it records their union, keeps every alternative, and
says least-privilege selection is deferred. Choosing is a policy decision, and policy is a
separate stage. Section 7 is where the choice gets made.

In [ ]:
for ambiguity in ir.ambiguities:
    print(f"{ambiguity.code}   blocking={ambiguity.blocking}")
    print(f"  at {ambiguity.source_pointer}")
    print(f"  {ambiguity.detail}\n")

default_response_classified_as_fault   blocking=False
  at openapi:#/paths/~1warehouses~1{warehouse_id}~1items/get/responses/default
  The OpenAPI `default` response was classified as a fault. Confirm during review that it is not a success case.

security_requirement_alternatives   blocking=False
  at openapi:#/paths/~1warehouses~1{warehouse_id}~1items/delete/security
  3 alternative security requirements were unioned into one over-approximated scope set. Least-privilege selection is deferred.



## 5. Planning

The baseline planner maps one operation to one tool, and exists for comparison. The
semantic planner renames, groups, projects arguments, reclassifies and omits. It records
a rationale and a confidence for each decision, because a surface a reviewer cannot argue
with is one they can only rubber-stamp.

Note the confidences: 1.0 for the clean read, 0.67 for the destructive operation, 0.5 for
the deprecated one. They are readiness signals counted from the specification, and the
rationale names which signals are missing.

In [ ]:
from api_mcp_compiler.planning.semantic import plan_semantic

plan = plan_semantic(ir)

for artifact in plan.artifacts:
    print(f"{artifact.name}")
    print(f"  {artifact.kind.value}, {artifact.risk.value}, confidence {artifact.confidence}")
    print(f"  from {artifact.source_operations}")

list_items_held_warehouse
  tool, read, confidence 1.0
  from ['listWarehouseItems']
permanently_remove_item_record_warehouse
  tool, destructive, confidence 0.6667
  from ['purgeWarehouseItems']
list_items_using_retired_v1
  resource, read, confidence 0.5
  from ['listWarehouseItemsLegacy']


## 6. The decisions that are not renames

Three kinds worth reading in full. A **projection** withholds an argument that is transport
rather than task. A **reclassification** turns an addressable read into a resource rather
than spending a tool slot on a lookup. An **omission** drops a deprecated operation, so
agent attention is not spent on a surface the provider intends to withdraw.

Each is a proposal with a confidence, not a fact. All three are reversible from an
overlay.

In [ ]:
for decision in plan.decisions:
    if decision.kind.value in {"project", "reclassify", "omit"}:
        print(f"{decision.kind.value}  {decision.target}  confidence {decision.confidence}")
        print(f"  {decision.rationale}\n")

project  listWarehouseItems  confidence 0.7
  Withholds page carry transport rather than task concerns. Each is optional and is left off the wire, so the service applies its own value. Confirm no caller needs to set them explicitly.

reclassify  listWarehouseItemsLegacy  confidence 0.6
  A read whose only inputs identify what to fetch is addressable, so a resource avoids spending a tool slot on a lookup.

omit  listWarehouseItemsLegacy  confidence 0.6
  The specification marks this operation deprecated, so exposing it spends agent attention on a surface the provider intends to withdraw.



## 7. Policy, generated separately from code

Policy generation is deliberately not part of code generation. A scope, an approval
requirement and a confirmation are governance claims; deriving them inside a template would
make them invisible to review.

Here is where the deferred scope choice from section 4 gets made, and where the useful
subtlety lives. **Fewest scopes is not the same as least privilege**: the scopeless admin
key is narrowest by count while granting the most, so a scoped alternative wins and the
rationale names what was rejected.

In [ ]:
from api_mcp_compiler.policy.synthesis import synthesize_policy

manifest = synthesize_policy(ir, plan)
policy = next(p for p in manifest.policies if p.tool_name.startswith("permanently_remove"))
reason = next(
    entry.rule.split(": ", 1)[1]
    for entry in policy.provenance
    if entry.rule.startswith("policy.rationale.scopes")
)

print(f"tool           {policy.tool_name}")
print(f"scopes         {policy.required_scopes}")
print(f"               {reason}")
print(f"approval       {policy.approval.value}")
print(f"confirmation   required={policy.confirmation.required} ttl={policy.confirmation.token_ttl_seconds}s")
print(f"               {policy.confirmation.effect_summary}")
print(f"rollback       {policy.rollback_guidance}")

tool           permanently_remove_item_record_warehouse
scopes         ['inventory.write']
               Narrowest of 3 alternatives, using inventoryOAuth. The union across alternatives would have granted inventory.admin, inventory.write; rejected inventoryOAuth(inventory.admin, inventory.write); inventoryAdminKey(no scopes).
approval       user_confirmation
confirmation   required=True ttl=300s
               permanently_remove_item_record_warehouse performs a destructive action that may not be reversible.
rollback       No automated compensation exists. Confirm the effect can be reversed manually before enabling in production.


## 8. What the compiler refuses

The destructive tool is generated, and it is not executable. It carries the blocker that
holds it.

It would have been easier to leave it out. Emitting it disabled is the deliberate choice: a
surface that silently dropped the operation would be indistinguishable from one where the
operation never existed, and nobody reviews an absence.

In [ ]:
from api_mcp_compiler.codegen.tools import generate_surface

surface = generate_surface(ir, plan, manifest)

for tool in surface.tools:
    blockers = [blocker.value for blocker in tool.blockers]
    print(f"{tool.name:42} {tool.emission.value:11} {blockers}")

held = next(tool for tool in surface.tools if tool.blockers)
print(f"\n{held.blocker_detail}")

list_items_held_warehouse                  executable  []
permanently_remove_item_record_warehouse   disabled    ['awaiting_approval']
list_items_using_retired_v1                executable  []

a write, destructive or privileged tool requires explicit human approval


## 9. The decision that releases it

Approval is granted over a class a person can reason about: every read, one group, one
risk class. It reports back what it covered. There is deliberately no flag that approves
a whole surface without naming what class of thing it belongs to.

The overlay is bound to the source digest. Approval granted against this specification does
not carry to a changed one; the restamp is a separate, deliberate act.

In [ ]:
from api_mcp_compiler.models import RiskClass
from api_mcp_compiler.planning.approval import approve

outcome = approve(plan, overlay=None, risk=RiskClass.DESTRUCTIVE, group=None, names=[])

print(f"approved   {outcome.approved}")
print(f"untouched  {outcome.untouched}")
print(f"bound to   {outcome.overlay.source_digest}\n")

approved_plan = plan_semantic(ir, outcome.overlay)
approved_manifest = synthesize_policy(ir, approved_plan)
approved_surface = generate_surface(ir, approved_plan, approved_manifest)

for tool in approved_surface.tools:
    blockers = [blocker.value for blocker in tool.blockers]
    print(f"{tool.name:42} {tool.emission.value:11} {blockers}")

approved   ['permanently_remove_item_record_warehouse']
untouched  ['list_items_held_warehouse', 'list_items_using_retired_v1']
bound to   sha256:3a0ae2f79c0d5b4b13e73266f61df6e77c6729e0a5d94a17716776ee9e2730a0

list_items_held_warehouse                  executable  []
permanently_remove_item_record_warehouse   executable  []
list_items_using_retired_v1                executable  []


## 10. The server

The emitted server binds to the MCP Python SDK. Read the registration below as the end of
the audit trail: `requires_confirmation` and `destructive` are the policy from section 7,
`max_output_bytes` is its output cap, `bindings` is where each argument goes on the wire,
and the confirmation token the runtime demands is bound to a digest of the arguments, so
confirming one call cannot authorise a different one.

In [ ]:
from api_mcp_compiler.codegen.mcp_server import emit_server

server = emit_server(ir, approved_surface, approved_manifest)

print(f"registered  {server.registered}")
print(f"withheld    {server.withheld}")
print(f"upstream    {server.base_url}")
print(f"{len(server.source.splitlines())} lines of Python\n")

body = server.source.splitlines()
start = next(i for i, line in enumerate(body) if "async def permanently_remove" in line)
print("\n".join(body[start - 1 : start + 14]))

registered  ['list_items_held_warehouse', 'permanently_remove_item_record_warehouse', 'list_items_using_retired_v1']
withheld    {}
upstream    https://inventory.example.invalid/v2
268 lines of Python

@mcp.tool(name='permanently_remove_item_record_warehouse', description='Permanently remove every item record for a warehouse. This removes data and cannot be undone.')
async def permanently_remove_item_record_warehouse(arguments: dict[str, Any]) -> dict[str, Any]:
    """Permanently remove every item record for a warehouse. This removes data and cannot be undone."""
    return await _invoke(
        tool_name='permanently_remove_item_record_warehouse',
        steps=[('DELETE', '/warehouses/{warehouse_id}/items', 'purgeWarehouseItems')],
        threading={},
        arguments=arguments,
        schema=_SCHEMAS['permanently_remove_item_record_warehouse'],
        bindings={'warehouse_id': ('path', 'warehouse_id', None)},
        requires_confirmation=True,
        confirmation_ttl_second

## What this was

Six artifacts, each validated against a versioned JSON Schema, each carrying the digest of
the specification it came from: IR, plan, policy manifest, surface, overlay, server.

Every stage here has a CLI equivalent (`inspect`, `plan`, `policy`, `generate`,
`approve-surface`, `serve`, `review`, `report`), so the same walk is available without
Python.

What this notebook is not is a review UI. Turning the gate into something a team operates
is a product, and a deliberately separate one.